#  Finance AI Agent using OpenAI Agents SDK + Ollama

## Objective

In this notebook, we will build a **Finance AI Agent** that can answer stock market queries using:

- **OpenAI Agents SDK**
- **Ollama** running locally
- **Yahoo Finance (yfinance)** for live market data

The agent is capable of:

- Retrieving the latest stock price
- Fetching analyst recommendations
- Calling Python functions as tools automatically
- Running completely on a local LLM through Ollama

---

In [1]:
import os
os.environ["OPENAI_BASE_URL"] = "http://localhost:11434/v1"
os.environ["OPENAI_API_KEY"] = "ollama"

In [2]:
import yfinance as yf

from agents import (
    Agent,
    Runner,
    function_tool,
    set_tracing_disabled,
)
import asyncio
import gradio as gr

D:\Tech Project\Finace AI Agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Disable Cloud Tracing

The OpenAI Agents SDK supports tracing through OpenAI.

Since we are running everything locally with Ollama, cloud tracing is unnecessary and is disabled.

In [3]:
# Disable OpenAI cloud tracing
set_tracing_disabled(True)

# Tool 1 — Get Current Stock Price

This tool retrieves the **latest closing price** for a stock ticker using Yahoo Finance.

### Input

Ticker Symbol

Example:

- TSLA
- AAPL
- MSFT

### Output

A formatted string containing the latest stock price.

In [4]:
# ============================================================
# Tool: Current Stock Price
# ============================================================

@function_tool
def get_stock_price(ticker: str) -> str:
    """
    Fetch the latest closing stock price
    for the given ticker symbol.
    """

    # Create Yahoo Finance object
    stock = yf.Ticker(ticker)

    # Download one day of price history
    history = stock.history(period="1d")

    # Extract latest closing price
    latest_close = history["Close"].iloc[-1]

    return f"The current price of {ticker} is ${latest_close:.2f}"

# Tool 2 — Analyst Recommendations

This tool retrieves analyst ratings from Yahoo Finance.

If recommendations are available, the latest 10 entries are returned.

Otherwise, a suitable message is displayed.

In [5]:
# ============================================================
# Tool: Analyst Recommendations
# ============================================================

@function_tool
def get_analyst_recommendations(ticker: str) -> str:
    """
    Retrieve analyst recommendations
    for the specified stock ticker.
    """

    # Create Yahoo Finance object
    stock = yf.Ticker(ticker)

    # Download analyst recommendation data
    recommendations = stock.recommendations

    # Handle missing recommendation data
    if recommendations is None or recommendations.empty:
        return f"No analyst recommendations found for {ticker}"

    # Return the latest 10 recommendations
    latest_recommendations = recommendations.tail(10)
    
    #return latest_recommendations.to_string()

    return latest_recommendations.to_markdown()


# Create the Finance Agent

The Finance Agent is responsible for:

- Understanding user queries
- Deciding when to use tools
- Calling the appropriate function
- Returning a natural language response

### Model

```
gemma4:31b-cloud
```

### Available Tools

- Current Stock Price
- Analyst Recommendations

In [6]:
# ============================================================
# Create the Finance AI Agent
# ============================================================

finance_agent = Agent(
    name="Finance Agent",
    instructions=(
        "You are a helpful finance assistant. "
        "Whenever appropriate, present structured information "
        "using tables."
    ),
    model="gemma4:31b-cloud",
    tools=[
        get_stock_price,
        get_analyst_recommendations,
    ],
)

# Run the Agent

We now send a user query to the agent.

The agent will:

1. Understand the question
2. Decide whether a tool is required
3. Execute the tool
4. Generate the final response

In [7]:
# ============================================================
# Execute the Agent
# ============================================================

result = await Runner.run(
    finance_agent,
    "What is the current stock price for TSLA?"
)

# Display the Response

The final output generated by the AI agent is displayed below.

In [8]:
# Print the final response

print(result.final_output)

The current stock price for TSLA is $397.28.


In [9]:
async def run_agent(user_input: str) -> str:
    result = await Runner.run(finance_agent, user_input)
    ai_output = result.final_output
    return ai_output


In [10]:
def chat_with_agent(user_input: str) -> str:
    final_output = asyncio.run(run_agent(user_input))
    return final_output

In [11]:
demo = gr.Interface(
    fn=chat_with_agent,
    inputs=gr.Textbox(label="Ask about a stock", placeholder="e.g. What is the current price of TSLA?"),
    outputs=gr.Textbox(label="Response", lines=12, max_lines=20),
    title="Finance AI Agent",
    description="Ask about stock prices or analyst recommendations."
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [12]:
if __name__ == "__main__":
    demo.launch()

AttributeError: 'TupleNoPrint' object has no attribute 'launch'